In [ ]:
#importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Global style — white background, black text everywhere
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    'black',
    'axes.labelcolor':   'black',
    'xtick.color':       'black',
    'ytick.color':       'black',
    'text.color':        'black',
    'grid.color':        '#cccccc',
    'font.family':       'sans-serif',
})

print("✓ All libraries loaded")


#loading data
uploaded = files.upload()
fname = list(uploaded.keys())[0]

xl = pd.ExcelFile(fname)
print(f"✓ Sheets found: {xl.sheet_names}")

# Edit this order to match your actual sheet names
STAGE_ORDER = ['zygote', 'early2cell', 'mid2cell', 'late2cell',
               '4cell', '8cell', '16cell', 'earlyblastula']

stages = [s for s in STAGE_ORDER if s in xl.sheet_names]
stages += [s for s in xl.sheet_names if s not in stages]
print(f"✓ Stage order used: {stages}")



# Average all replicate columns → one mean expression per gene per stage
# Duplicate gene names are merged by averaging

stage_dfs = {}
for stage in stages:
    df = xl.parse(stage)
    df.columns = df.columns.astype(str).str.strip()
    df = df.dropna(subset=['Gene'])

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    is_x_col = df.groupby('Gene')['is_X'].first() if 'is_X' in df.columns else None
    df = df.groupby('Gene')[numeric_cols].mean()  # merge duplicates by mean

    if is_x_col is not None:
        df['is_X'] = is_x_col

    print(f"  {stage}: {len(df)} unique genes (duplicates merged)")
    stage_dfs[stage] = df

expr_matrix = pd.DataFrame({
    stage: stage_dfs[stage].drop(columns=['is_X'], errors='ignore')
                           .select_dtypes(include=np.number)
                           .mean(axis=1)
    for stage in stages
})

# Keep only X-linked genes
x_genes = set()
for stage, df in stage_dfs.items():
    if 'is_X' in df.columns:
        x_genes.update(df[df['is_X'] == True].index.tolist())

expr_matrix = expr_matrix[expr_matrix.index.isin(x_genes)]
expr_matrix = expr_matrix.fillna(0)

print(f"✓ Matrix: {expr_matrix.shape[0]} X-linked genes × {expr_matrix.shape[1]} stages")
print(expr_matrix.round(3).head())


#
# log2(RPKM FC + 1): 0 = no change, not absent → keep zeros
# Only remove completely flat genes (zero variance = uninformative)

gene_std = expr_matrix.std(axis=1)
expr_filtered = expr_matrix[gene_std > 0]

cv = expr_filtered.std(axis=1) / (expr_filtered.abs().mean(axis=1) + 1e-9)
CV_THRESHOLD = 0.1
expr_filtered = expr_filtered[cv > CV_THRESHOLD]

print(f"✓ Genes after filtering: {expr_filtered.shape[0]}")

expr_scaled = pd.DataFrame(
    StandardScaler().fit_transform(expr_filtered.T).T,
    index=expr_filtered.index,
    columns=expr_filtered.columns
)
print("✓ Z-score normalisation done")


# Find Optimal K (Elbow + Silhouette)
inertias, sil_scores = [], []
K_RANGE = range(2, 9)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(expr_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(expr_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_RANGE, inertias, 'o-', color='steelblue', lw=2, ms=7)
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method', fontweight='bold')
ax1.set_xticks(K_RANGE)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.plot(K_RANGE, sil_scores, 'o-', color='seagreen', lw=2, ms=7)
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score  (higher = better)', fontweight='bold')
ax2.set_xticks(K_RANGE)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

best_k = list(K_RANGE)[np.argmax(sil_scores)]
ax2.axvline(best_k, color='crimson', ls='--', lw=1.5, label=f'Best K = {best_k}')
ax2.legend()

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Suggested K = {best_k}  (override in Cell 6 if needed)")


# ── CELL 6: K-Means Clustering ───────────────────────────────
K = best_k   # ← override manually if needed, e.g. K = 4

km = KMeans(n_clusters=K, random_state=42, n_init=50)
cluster_labels = km.fit_predict(expr_scaled) + 1  # 1-indexed
expr_scaled['Cluster'] = cluster_labels

for c in sorted(expr_scaled['Cluster'].unique()):
    n = (expr_scaled['Cluster'] == c).sum()
    print(f"  Cluster {c}: {n} genes")



# Within each k-means cluster, reorder genes by hierarchical linkage

CLUSTER_COLORS = plt.cm.Set2(np.linspace(0, 1, K))

ordered_genes, cluster_boundaries, pos = [], [], 0

for c in range(1, K + 1):
    gene_list = expr_scaled[expr_scaled['Cluster'] == c].index.tolist()
    sub = expr_scaled.loc[gene_list].drop(columns='Cluster')

    if len(gene_list) > 2:
        Z = linkage(sub, method='ward')
        order = dendrogram(Z, no_plot=True)['leaves']
        gene_list = [gene_list[i] for i in order]

    ordered_genes.extend(gene_list)
    cluster_boundaries.append((pos, pos + len(gene_list), c))
    pos += len(gene_list)

heatmap_data = expr_scaled.drop(columns='Cluster').loc[ordered_genes]

n_genes = len(ordered_genes)
fig_h = min(max(10, n_genes * 0.15), 40)   # cap at 40 inches
fig_w = max(14, len(stages) * 1.5)

fig, (ax_bar, ax_heat) = plt.subplots(
    1, 2, figsize=(fig_w, fig_h),
    gridspec_kw={'width_ratios': [0.03, 1]},
    facecolor='white'
)

# Cluster colour sidebar
for start, end, c in cluster_boundaries:
    ax_bar.barh(range(start, end), 1, color=CLUSTER_COLORS[c - 1], left=0)
ax_bar.set_xlim(0, 1)
ax_bar.set_ylim(-0.5, n_genes - 0.5)
ax_bar.invert_yaxis()
ax_bar.axis('off')
ax_bar.set_facecolor('white')

# Heatmap
cmap = sns.diverging_palette(240, 10, s=75, l=45, as_cmap=True)
sns.heatmap(
    heatmap_data, ax=ax_heat,
    cmap=cmap, center=0, vmin=-2.5, vmax=2.5,
    xticklabels=True,
    yticklabels=True if n_genes <= 150 else False,
    linewidths=0,
    cbar_kws={
        'label':  'Z-score of log2(FC+1)',
        'shrink': 0.15,
        'aspect': 10,
        'pad':    0.02
    }
)
ax_heat.set_facecolor('white')
ax_heat.tick_params(axis='y', labelsize=max(4, 8 - n_genes // 30),
                    colors='black', length=0)
ax_heat.tick_params(axis='x', colors='black', rotation=45, labelsize=9)
ax_heat.set_ylabel('')
ax_heat.set_xlabel('Developmental Stage', color='black')

# Cluster labels + dividers
for start, end, c in cluster_boundaries:
    ax_heat.annotate(
        f' C{c}',
        xy=(len(stages) + 0.1, (start + end) / 2),
        xycoords='data', fontsize=8, va='center', fontweight='bold',
        color=mcolors.to_hex(CLUSTER_COLORS[c - 1]),
        annotation_clip=False
    )
    if end < n_genes:
        ax_heat.axhline(end, color='white', lw=2)

cbar = ax_heat.collections[0].colorbar
cbar.ax.tick_params(colors='black', labelsize=8)
cbar.set_label('Z-score of log2(FC+1)', color='black', fontsize=8)

plt.suptitle(
    f'X-linked Gene Expression · K={K} Clusters\nlog2(RPKM FC+1), Z-scored across stages',
    color='black', fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('clustering_heatmap.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Heatmap saved")



# Mean ± SD trajectory per cluster
# Above 0 = upregulated vs gene mean, below 0 = downregulated

fig, axes = plt.subplots(1, K, figsize=(4 * K, 4), sharey=True, facecolor='white')
if K == 1:
    axes = [axes]

for c in range(1, K + 1):
    ax = axes[c - 1]
    ax.set_facecolor('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ['bottom', 'left']:
        ax.spines[spine].set_color('black')

    gene_list = expr_scaled[expr_scaled['Cluster'] == c].index.tolist()
    sub = expr_scaled.loc[gene_list].drop(columns='Cluster')

    # Individual gene traces (faint)
    for _, row in sub.iterrows():
        ax.plot(range(len(stages)), row.values,
                color=CLUSTER_COLORS[c - 1], alpha=0.12, lw=0.8)

    # Mean ± SD
    mean, std = sub.mean(), sub.std()
    ax.plot(range(len(stages)), mean.values,
            color=CLUSTER_COLORS[c - 1], lw=2.5, zorder=5)
    ax.fill_between(range(len(stages)), mean - std, mean + std,
                    color=CLUSTER_COLORS[c - 1], alpha=0.2)

    ax.axhline(0, color='grey', ls='--', lw=0.8)
    ax.set_xticks(range(len(stages)))
    ax.set_xticklabels(stages, rotation=45, ha='right', fontsize=7, color='black')
    ax.tick_params(axis='y', colors='black')
    ax.set_title(f'Cluster {c}  ({len(gene_list)} genes)',
                 color=mcolors.to_hex(CLUSTER_COLORS[c - 1]),
                 fontsize=9, fontweight='bold')

axes[0].set_ylabel('Z-score of log2(FC+1)', color='black', fontsize=9)
plt.suptitle(
    'Mean Expression Profile per Cluster\n(above 0 = upregulated, below 0 = downregulated)',
    color='black', fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('cluster_profiles.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Profiles saved")


# Export Results
results = expr_scaled[['Cluster']].copy()
for stage in stages:
    results[f'{stage}_mean_log2FC'] = expr_filtered.loc[results.index, stage]

results.index.name = 'Gene'
results.to_csv('cluster_assignments.csv')
print("✓ Saved: cluster_assignments.csv")

for c in range(1, K + 1):
    genes = results[results['Cluster'] == c].index.tolist()
    print(f"\nCluster {c}  ({len(genes)} genes):")
    print(", ".join(genes[:20]) + (" ..." if len(genes) > 20 else ""))

for f in ['cluster_assignments.csv', 'clustering_heatmap.png',
          'cluster_profiles.png', 'elbow_silhouette.png']:
    try:
        files.download(f)
    except:
        pass

print("\n✓ Done — all files downloaded")